In [112]:
!pip install pypdf sentence-transformers pinecone
# optional but recommended for better tokenization/splitting:
!pip install nltk
!python -m nltk.downloader punkt


<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [113]:
import os
import uuid
import math
from typing import List, Dict, Any, Optional

from pypdf import PdfReader
import tiktoken
from sentence_transformers import SentenceTransformer

# pinecone import (simple usage)
import pinecone

In [114]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim
TOKEN_ENCODING = "cl100k_base"

MAX_TOKENS = 150
OVERLAP_TOKENS = 30
EMBED_BATCH_SIZE = 64
UPSERT_BATCH_SIZE = 64


In [115]:
!pip install PyMuPDF tiktoken

import fitz  # PyMuPDF
import tiktoken
import re

def extract_text_from_pdf(pdf_path):
    """Extract text using PyMuPDF with better word spacing"""
    text = ""

    # Open PDF
    doc = fitz.open(pdf_path)

    for page in doc:
        # Extract text with word-level detail
        text += page.get_text("text") + "\n"

    doc.close()

    # Clean up the text
    text = text.replace("\x00", "")

    # Fix common issues
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)  # Split camelCase
    text = re.sub(r'([.,!?;:])([A-Za-z])', r'\1 \2', text)  # Space after punctuation
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
    text = re.sub(r' +([.,!?;:])', r'\1', text)  # Remove space before punctuation

    # Fix specific patterns in academic papers
    text = re.sub(r'(\d)([A-Z][a-z])', r'\1 \2', text)  # "2014English" -> "2014 English"
    text = re.sub(r'([a-z])(\d)', r'\1 \2', text)  # "train28" -> "train 28"

    text = text.strip()
    return text


def extract_text_blocks(pdf_path):
    """Alternative: Extract text in blocks for better structure"""
    text = ""

    doc = fitz.open(pdf_path)

    for page in doc:
        blocks = page.get_text("blocks")  # Get text blocks
        for block in blocks:
            if block[6] == 0:  # Text block (not image)
                block_text = block[4].strip()
                if block_text:
                    text += block_text + " "

    doc.close()

    # Clean up
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([.,!?;:])([A-Za-z])', r'\1 \2', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text


def chunk_text_by_tokens(text, chunk_size=512, overlap=50):
    """Split text into overlapping chunks"""

    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)

        start += (chunk_size - overlap)

        if end == len(tokens):
            break

    return chunks


# Usage
pdf_path = "/content/paper.pdf"

# Try both methods
print("Method 1: Basic text extraction")
text1 = extract_text_from_pdf(pdf_path)
print(text1[:500])

print("\n" + "="*80 + "\n")

print("Method 2: Block-based extraction")
text2 = extract_text_blocks(pdf_path)
print(text2[:500])



Method 1: Basic text extraction
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research usz@google. com Llion Jones∗ Google Research llion@google. com Aidan N. Gomez∗† University of Toronto aidan@cs. toronto. edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google. com Illia Polosukhin∗‡ illia. polosukhin@gmail. com Abstract The dominant sequence transduction models are based on complex re


Method 2: Block-based extraction
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research usz@google. com Llion Jones∗ Google Research llion@google. com Aidan N. Gomez∗† University of Toronto aidan@cs. toronto. edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google. com Illia Polosukhin∗‡ illia. polosukhin@gmail. com Abstra

In [116]:
# Use the better one
text = text1  # or text2, whichever looks better

# Create chunks
chunks = chunk_text_by_tokens(text, chunk_size=512, overlap=50)

print(f"\n\nTotal text length: {len(text)} characters")
print(f"Number of chunks: {len(chunks)}")
print(f"\nFirst chunk preview:\n{chunks[0][:300]}...")



Total text length: 32780 characters
Number of chunks: 18

First chunk preview:
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research usz@google. com Llion Jones∗ Google Research llion@google. com Aidan N. Gomez∗† University of Toront...


In [117]:
len(chunks)

18

In [118]:
chunks[0]

'Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research usz@google. com Llion Jones∗ Google Research llion@google. com Aidan N. Gomez∗† University of Toronto aidan@cs. toronto. edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google. com Illia Polosukhin∗‡ illia. polosukhin@gmail. com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring signiﬁcantly less time to train. 

In [119]:
from sentence_transformers import SentenceTransformer

def embed_chunks(chunks, model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=True)
    return embeddings


In [120]:
embeddings=embed_chunks(chunks, model_name="all-MiniLM-L6-v2")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [121]:
len(embeddings)

18

In [122]:
embeddings = embeddings.tolist()
print(f"Type of embeddings after conversion: {type(embeddings)}")
print(f"First embedding after conversion: {embeddings[0][:5]}...") # Displaying first 5 elements of the first embedding

Type of embeddings after conversion: <class 'list'>
First embedding after conversion: [-0.08127140998840332, -0.12509962916374207, 0.01718575693666935, -0.0258807186037302, 0.016683906316757202]...


In [123]:
embeddings

[[-0.08127140998840332,
  -0.12509962916374207,
  0.01718575693666935,
  -0.0258807186037302,
  0.016683906316757202,
  0.019419878721237183,
  -0.057994186878204346,
  0.007092613726854324,
  0.09432998299598694,
  -0.06109823286533356,
  0.027653219178318977,
  -0.05657521262764931,
  -0.05618331581354141,
  0.0017161432188004255,
  -0.054990917444229126,
  0.005669428966939449,
  0.040857378393411636,
  0.06816880404949188,
  -0.07678210735321045,
  -0.08850826323032379,
  0.012415971606969833,
  0.0209802333265543,
  0.017359336838126183,
  0.021719640120863914,
  0.00808352418243885,
  0.02889249473810196,
  -0.06428485363721848,
  -0.06043645367026329,
  -0.006612163502722979,
  0.0004422613128554076,
  0.040043413639068604,
  -0.04139484837651253,
  -0.034718163311481476,
  0.0811622366309166,
  -0.051018599420785904,
  0.1084047183394432,
  -0.08498164266347885,
  -0.018717851489782333,
  -0.0030366249848157167,
  -0.07355861365795135,
  0.006905333139002323,
  0.00783826690167

In [124]:
chunks

['Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research usz@google. com Llion Jones∗ Google Research llion@google. com Aidan N. Gomez∗† University of Toronto aidan@cs. toronto. edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google. com Illia Polosukhin∗‡ illia. polosukhin@gmail. com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring signiﬁcantly less time to train.

In [131]:
len(embeddings[0])

384

In [133]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(api_key="")

# Create index if it doesn't exist
index_name = "test"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(embeddings[0]),  # Length of first embedding
        metric="cosine",
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

# Connect to index
index = pc.Index(index_name)

# Prepare and upsert vectors
vectors = [
    (f"chunk_{i}", embedding, {"text": chunk})  # No .tolist() needed, chunks are already strings
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings))
]

index.upsert(vectors=vectors)
print("Upsert completed!")

Upsert completed!


In [135]:
vectors[0]

('chunk_0',
 [-0.08127140998840332,
  -0.12509962916374207,
  0.01718575693666935,
  -0.0258807186037302,
  0.016683906316757202,
  0.019419878721237183,
  -0.057994186878204346,
  0.007092613726854324,
  0.09432998299598694,
  -0.06109823286533356,
  0.027653219178318977,
  -0.05657521262764931,
  -0.05618331581354141,
  0.0017161432188004255,
  -0.054990917444229126,
  0.005669428966939449,
  0.040857378393411636,
  0.06816880404949188,
  -0.07678210735321045,
  -0.08850826323032379,
  0.012415971606969833,
  0.0209802333265543,
  0.017359336838126183,
  0.021719640120863914,
  0.00808352418243885,
  0.02889249473810196,
  -0.06428485363721848,
  -0.06043645367026329,
  -0.006612163502722979,
  0.0004422613128554076,
  0.040043413639068604,
  -0.04139484837651253,
  -0.034718163311481476,
  0.0811622366309166,
  -0.051018599420785904,
  0.1084047183394432,
  -0.08498164266347885,
  -0.018717851489782333,
  -0.0030366249848157167,
  -0.07355861365795135,
  0.006905333139002323,
  0.00

In [138]:
# Step 1: Create embedding for your query
query_text = "explain self attention and how is it different from deep learning?"
query_embedding = embed_chunks([query_text])[0]  # Adjust based on your embedding model

# Step 2: Query Pinecone
results = index.query(
    vector=query_embedding.tolist(),
    top_k=5,  # Number of closest matches to return
    include_metadata=True  # Include the text chunks
)

# Step 3: Display results
for i, match in enumerate(results['matches']):
    print(f"\nRank {i+1}:")
    print(f"Score: {match['score']:.4f}")
    print(f"ID: {match['id']}")
    print(f"Text: {match['metadata']['text'][:200]}...")  # First 200 chars

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Rank 1:
Score: 0.5964
ID: chunk_2
Text:  of the Extended Neural GPU [20], Byte Net [15] and Conv S2S [8], all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and...

Rank 2:
Score: 0.5007
ID: chunk_8
Text:  1, a self-attention layer connects all positions with a constant number of sequentially executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of computational c...

Rank 3:
Score: 0.4831
ID: chunk_3
Text: attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is ...

Rank 4:
Score: 0.4804
ID: chunk_0
Text: Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google. com Noam Shazeer∗ Google Brain noam@google. com Niki Parmar∗ Google Research nikip@google. com Jakob Uszkoreit∗ Google Research ...

Rank 5:
Score: 0.46

In [139]:
results

QueryResponse(matches=[{'id': 'chunk_2',
 'metadata': {'text': ' of the Extended Neural GPU [20], Byte Net [15] and '
                      'Conv S2S [8], all of which use convolutional neural '
                      'networks as basic building block, computing hidden '
                      'representations in parallel for all input and output '
                      'positions. In these models, the number of operations '
                      'required to relate signals from two arbitrary input or '
                      'output positions grows in the distance between '
                      'positions, linearly for Conv S2S and logarithmically '
                      'for Byte Net. This makes it more difﬁcult to learn '
                      'dependencies between distant positions [11]. In the '
                      'Transformer this is reduced to a constant number of '
                      'operations, albeit at the cost of reduced effective '
                      'resolution du